# CAP Sleep Database: model evaluation and interpretation

This notebook reads only validated pipeline outputs. Primary model inputs exclude clinician annotations. The cohort has one recording per participant; all scores below are subject-level out-of-fold estimates from shared folds. These retrospective results are exploratory and are not clinical performance estimates.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'results').exists():
    root = root.parent
results = root / 'results'
with (results / 'schema_validation.json').open(encoding='utf-8') as stream:
    schema = json.load(stream)
assert schema['status'] == 'validated'
with (results / 'models' / 'primary' / 'metrics.json').open(encoding='utf-8') as stream:
    metrics = json.load(stream)
metrics['models']


In [ ]:
score_table = pd.DataFrame({
    name: {'Balanced accuracy': values['balanced_accuracy'], 'Macro-F1': values['macro_f1']}
    for name, values in metrics['models'].items()
}).T
score_table.plot.bar(rot=0, ylim=(0, 1), ylabel='Out-of-fold score', title='Primary physiology-only models')
plt.tight_layout()


In [ ]:
qc = pd.read_csv(results / 'qc' / 'recording_qc.csv')
display(qc.groupby('label')[['recognized_channel_count', 'excluded_channel_count']].describe())
annotations = pd.read_csv(results / 'qc' / 'annotation_qc.csv')
display(annotations['status'].value_counts(dropna=False))


In [ ]:
cap_summary_path = results / 'cap_group_summary.csv'
if cap_summary_path.exists():
    cap_summary = pd.read_csv(cap_summary_path, index_col=0)
    ax = cap_summary['cap_rate_percent_mean'].plot.bar(
        yerr=cap_summary['cap_rate_percent_std'],
        ylabel='CAP rate (%)', title='Clinician-scored CAP rate by diagnosis'
    )
    ax.set_xlabel('Diagnostic group')
    plt.tight_layout()


Interpret class-wise errors and QC before making claims about modality value. The models differ only in modality feature columns; this observational comparison does not imply causal benefit. Report class support, confidence intervals or repeated validation in any final scientific report, and discuss the small, imbalanced, single-database cohort as a limitation. CAP descriptions and the CAP-only experiment are separate secondary analyses.
